In [1]:
!pip install -U langchain-community
!pip install pypdf

Lectura PDF

In [2]:
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Cargar el PDF desde ,las carpetas de Drive
pdf_path = "./el principito.pdf"
loader = PyPDFLoader(pdf_path)
documents = loader.load()

# Extraer el texto de cada página
text = "\n".join([doc.page_content for doc in documents])

# Chunks

In [3]:
# Dividir en fragmentos (chunks)
text_splitter = RecursiveCharacterTextSplitter(chunk_size= 500, chunk_overlap = 200)

chunks = text_splitter.split_text(text)  # revisar como usar .split_docuement para que sea compatible con chromaDB

# Transformar texto y vectores

In [4]:
from sentence_transformers import SentenceTransformer

def text_to_vector(text):

    # Cargar el modelo de embeddings
    model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

    # Convertir el texto en un vector numérico
    vector = model.encode(text)

    return vector

def vector_to_text(vector):
  # Cargar modelo
  model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

  return text

frase_buscar = "Lo esencial es invisible a los ojos."

frase_vector = text_to_vector(frase_buscar)

frase_vector = frase_vector.astype(float).tolist()

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


# Chunks en vectores

In [5]:
chunk_vectors = [text_to_vector(chunk) for chunk in chunks]
print(len(chunk_vectors))

264


# Guardar Chunks

In [6]:
# import numpy as np

# # Convertir a un array de NumPy y guardar
# np.save("chunks.npy", np.array(chunk_vectors))

# # Leer desde el archivo .npy
# loaded_vectors = np.load("chunks.npy")

# print("Vectores cargados:", loaded_vectors)

# Instalar SWIG

In [7]:
!apt-get install swig

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
swig is already the newest version (4.0.2-1ubuntu1).
0 upgraded, 0 newly installed, 0 to remove and 30 not upgraded.


# Ball Tree

## Headers

### Se definen qué métodos son públicos y privados

In [8]:
%%file ball_tree.h

#ifndef BALL_TREE_H
#define BALL_TREE_H

#include <vector>

class BallTree {
private:
    struct Compare {
        int depth;
        Compare(int d) : depth(d) {}

        bool operator()(const std::vector<double>& a, const std::vector<double>& b) const {
            return a[depth % a.size()] < b[depth % a.size()];
        }
    };

    struct Node {
        int index;
        std::vector<double> point;
        Node* left;
        Node* right;
    };

    Node* root;
    std::vector<std::vector<double> > data;

    // Función para construir el árbol
    Node* build_tree(int left, int right, int depth);

    // Función para encontrar el vecino más cercano
    void nearest_neighbor(Node* node, const std::vector<double>& target, Node*& best, double& best_dist, int depth);

public:
    BallTree(); // Constructor

    // Construye el árbol a partir de un conjunto de puntos
    void build(const std::vector<std::vector<double> >& points);

    // Encuentra el punto más cercano en el árbol
    int find_nearest(const std::vector<double>& target);
};

#endif // BALL_TREE_H


Overwriting ball_tree.h


## Class

### Clase encargada de la lógica

In [9]:
%%file ball_tree.cpp
#include "ball_tree.h"
#include <cmath>
#include <limits>
#include <algorithm>

// Función para calcular la distancia euclidiana
double euclidean_distance(const std::vector<double>& a, const std::vector<double>& b) {
    double sum = 0.0;
    for (size_t i = 0; i < a.size(); ++i) {
        sum += (a[i] - b[i]) * (a[i] - b[i]);
    }
    return std::sqrt(sum);
}

// Constructor de la clase BallTree
BallTree::BallTree() : root(nullptr) {}

// Método para construir el árbol
void BallTree::build(const std::vector<std::vector<double> >& points) {
    this->data = points;
    this->root = build_tree(0, data.size() - 1, 0);
}

// Método privado para construir el árbol
BallTree::Node* BallTree::build_tree(int left, int right, int depth) {
    if (left > right) return nullptr;

    int mid = (left + right) / 2;
    std::nth_element(data.begin() + left, data.begin() + mid, data.begin() + right + 1, BallTree::Compare(depth));

    Node* node = new Node();
    node->index = mid;
    node->left = build_tree(left, mid - 1, depth + 1);
    node->right = build_tree(mid + 1, right, depth + 1);

    return node;
}

// Método privado para encontrar el vecino más cercano
void BallTree::nearest_neighbor(Node* node, const std::vector<double>& target, Node*& best, double& best_dist, int depth) {
    if (!node) return;

    double dist = euclidean_distance(target, node->point);
    if (dist < best_dist) {
        best_dist = dist;
        best = node;
    }

    int axis = depth % target.size();
    Node* next = target[axis] < node->point[axis] ? node->left : node->right;
    Node* other = (next == node->left) ? node->right : node->left;

    nearest_neighbor(next, target, best, best_dist, depth + 1);

    if (std::abs(target[axis] - node->point[axis]) < best_dist) {
        nearest_neighbor(other, target, best, best_dist, depth + 1);
    }
}

// Método público para encontrar el punto más cercano
int BallTree::find_nearest(const std::vector<double>& target) {
    Node* best = nullptr;
    double best_dist = std::numeric_limits<double>::max();
    nearest_neighbor(root, target, best, best_dist, 0);
    return best ? best->index : -1;
}


Overwriting ball_tree.cpp


## Interfaz

In [10]:
%%file ball_tree.i
%module ball_tree
%{
#include "ball_tree.h"
%}

%include "std_vector.i"
%template(VectorDouble) std::vector<double>;
%template(VectorVectorDouble) std::vector<std::vector<double>>;

%include "ball_tree.h"

// Exponer la clase BallTree a Python
%feature("director") BallTree;

// Funcion para retornar indice
%extend BallTree {
    int find_nearest(const std::vector<double>& target) {
        return $self->find_nearest(target);
    }
}


Overwriting ball_tree.i


# Ejecutar SWIG

In [11]:
!swig -c++ -python ball_tree.i

In [12]:
!g++ -O2 -fPIC -c ball_tree.cpp

In [13]:
!g++ -O2 -fPIC -c ball_tree_wrap.cxx -I/usr/include/python3.10

In [14]:
!g++ -shared ball_tree.o ball_tree_wrap.o -o _ball_tree.so

# Usar DLL

In [ ]:
import numpy as np
import ball_tree

tree = ball_tree.BallTree()

vector_float = np.array(chunk_vectors, dtype=np.float32)

vector_float = vector_float.tolist()

tree.build(vector_float)

chunk_similar = tree.find_nearest(frase_vector)

print(chunk_similar)

